# Quick sanity check of a trained adapter
Loads a run's LoRA adapter on the pre-tuned model and answers one domain, one tool-calling and one
general prompt. Uses the last run trained on this machine; set `RUN_NAME` to check another run
(its adapter is downloaded from S3 if it is not here).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('../..').resolve()))
from unsloth import FastLanguageModel          # before transformers, so Unsloth can patch it
import torch
from sft_common.adapter import point_adapter_at_base
from sft_common.chat_format import ChatFormat, split_think
from sft_common.config import load_config
from sft_common.pretuned import ensure_local_model
from sft_common.runs import existing_run
from sft_common.tool_calls import parse_calls

RUN_NAME = None                                # None = the last run started on this machine
cfg = load_config()
run = existing_run(cfg, RUN_NAME)
if not (run.adapter_dir / 'adapter_config.json').exists():
    run.store.download_dir('final_adapter', run.adapter_dir)   # adapter trained elsewhere
point_adapter_at_base(run.adapter_dir, ensure_local_model(cfg))  # base = this machine's pre-tuned copy

model, tok = FastLanguageModel.from_pretrained(str(run.adapter_dir), max_seq_length=cfg['model']['max_seq_length'],
                                               dtype=torch.bfloat16)
FastLanguageModel.for_inference(model)
fmt = ChatFormat(tok)
print('run:', run.name)

In [ ]:
def ask(messages, thinking=False, max_new_tokens=1024):
    """Generate an answer; returns (reasoning, answer, tool calls parsed from the answer)."""
    inputs = tok(text=fmt.prompt(messages, thinking=thinking), return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
    text = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    reasoning, answer = split_think(text)
    return reasoning, answer, parse_calls(answer)

# Tool definitions go in the system prompt as text, exactly like the training data.
weather_system = ('You are a helpful assistant with access to the following functions. Use them if required -
'
                  '{"name": "get_weather", "description": "Get the weather for a city", '
                  '"parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}')
tests = {
    'domain (thinking on)': ([{'role': 'system', 'content': 'You are a finance and investment assistant.'},
                              {'role': 'user', 'content': 'Explain how portfolio beta differs from standard deviation.'}], True),
    'tool call':            ([{'role': 'system', 'content': weather_system},
                              {'role': 'user', 'content': 'What is the weather in Paris?'}], False),
    'general':              ([{'role': 'user', 'content': 'Give me three tips for writing clear emails.'}], False),
}
for name, (msgs, thinking) in tests.items():
    reasoning, answer, calls = ask(msgs, thinking, max_new_tokens=2048 if thinking else 512)
    print(f'--- {name} ---')
    if reasoning:
        print('[reasoning]', reasoning[:600])
    print('[answer]', answer[:1200])
    print('[tool calls]', calls)